## End-to-end demo of CREST using MNIST data 

The entire notebook should run in < 60 seconds, and use ~600MB of memory.

In [16]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [17]:
# Save the start time
import time
start_time = time.time()

# Allow crest to be imported
%cd -q ../..

#### Load MNIST

In [18]:
import tensorflow as tf
import numpy as np

# Load data and define constants
xy,xy_test = tf.keras.datasets.mnist.load_data() 
xy_train   = [i[:50000]for i in xy]
xy_valid   = [i[50000:]for i in xy]
n_classes  = 10
patch_size = 28

#### Create the datasets

In [19]:
from crest.src.data.loading import StructuredDataset

# Set input/output key labels
INP = 'image'
OUT = 'class'

# Define the train/test datasets, as well as the array labels
ds_train = StructuredDataset(*xy_train, labels=[INP, OUT])
ds_valid = StructuredDataset(*xy_valid, labels=[INP, OUT])
ds_test  = StructuredDataset(*xy_test,  labels=[INP, OUT])

#### Create the batchers

In [20]:
from crest.src.data import Batcher

# Note: workers=0 is faster for this, but may cause issues in a notebook
# since the background threads will run in the main process 
batch_kwargs = {
    'batch_size' : 200,            # 200 samples in each batch
    'features'   : [[INP], [OUT]], # [{'image': ...}, {'class': ...}]
    'repeat'     : True,           # Yield indefinitely
    'workers'    : 0               # Number of background processes
}
batch_train = Batcher(ds_train, **batch_kwargs)
batch_kwargs['shuffle'] = False    # Turn shuffle off for prediction
batch_valid = Batcher(ds_valid,  **batch_kwargs)
batch_test  = Batcher(ds_test ,  **batch_kwargs)

In [26]:
a = next(batch_test)

dict_keys(['class'])

#### Create the CREST graph

In [ ]:
from tensorflow.keras.layers import Conv2D, Dropout, Flatten, Dense, Lambda
from crest.src.model import HierarchalTensorGraph as HTG

def crest_layer(layer):
    """ Wrap a tensorflow layer in a HierarchalTensorGraph """
    graph = HTG(lambda d: {OUT: layer(d[INP])}, name=layer.name)
    #set inputs/outputs
    graph.inputs = {INP:None}
    graph.outputs = {OUT:None}
    return graph

# Define the tensorflow layers, and convert to HTG objects
layers = [crest_layer(layer) for layer in [
    Lambda(lambda x: tf.cast(x, tf.float32)),         # Cast dtype
    Lambda(lambda x: tf.expand_dims(x, axis=-1)),     # Add feature dimension
    Conv2D(10, 3, padding='same', activation='relu'), # 28x28x1 -> 28x28x10
    Flatten(),                                        # 28x28x10 -> 7840
    Dropout(0.3),                                     # 30% of data dropped
    Dense(n_classes, activation='softmax'),           # Class probabilities
]]

# Rename outputs to use as inputs (except the last layer)
for i in layers[:-1]:
    i.rename_io(outputs_map={OUT:INP})
    
# Create the CREST graph: ensure tensor shapes are specified for the inputs and outputs
graph = HTG(
    name='MNIST',
    inputs={INP : tf.TensorSpec(shape=[None,patch_size, patch_size])},
    outputs={OUT : tf.TensorSpec(shape=[None,10])}
)
print(list(map(graph.add_edge, ['input']+layers, layers+['output'])))

#### Build, train, and test using Model

In [ ]:
from crest import Model

build_kwargs= {
    'optimizer' : 'Adam', 
    'loss'      : 'sparse_categorical_crossentropy', 
    'metrics'   : 'sparse_categorical_accuracy'
}

model = Model(graph)
model.build(**build_kwargs)
model.fit(batch_train, **{
    'validation_data'  : batch_valid,
    'steps_per_epoch'  : 250, # 50k samples / 200 samples per batch
    'validation_steps' : 50,  # 10k samples / 200 samples per batch
    'epochs'           : 5,
    'verbose'          : 1,
})


In [ ]:
# Evaluate the model on the test set
accuracy = model.evaluate(batch_test,verbose=False,return_dict=True)
print(f'Test performance metrics: {accuracy}')

Accuracy on the full test set should be > ~97%.

In [ ]:
# Close the batchers
batch_train.close()
batch_valid.close()
batch_test.close()

### Make predictions

In [ ]:
x_pred,_ = xy_test  # Use images from the testing set
ds_pred = StructuredDataset(*[x_pred], labels=[INP])
batch_pred = Batcher(ds_pred, **{
    'batch_size' : 200,
    'features'   : [INP],
    'repeat'     : True,
    'workers'    : 0,
})

In [ ]:
# Set kwargs for Model.predict() (can be the same as Keras.predict)
pred_kwargs = {
    'batch_size' : 200,
    'verbose'    : False
}

In [ ]:
# Generate predictions on the whole test set
pred_kwargs['steps'] = 10000//pred_kwargs['batch_size']
batch_pred = Batcher(ds_pred, **{  # Reset batch iterator
    'batch_size' : 200,
    'features'   : [INP],
    'repeat'     : True,
    'workers'    : 0,
    'shuffle'    : False
}) 
model_pred_batch = model.predict(batch_pred, **pred_kwargs) # Using a batcher
model_pred_dataset = model.predict(ds_pred,**pred_kwargs)   # Using a dataset
print('Difference:', (model_pred_batch['class'] - model_pred_dataset['class']).sum())

Adding auxillary output like lat and lon when making a prediction

In [ ]:
# Combine some fictous auxillary data with testing input for prediction
x,_ = xy_test
lat = np.arange(len(x),dtype=float)
lon = np.arange(len(x),dtype=float)
x_pred = (lat,lon,x)

In [ ]:
# Create the prediction data and make predictions
ds_pred = StructuredDataset(*x_pred, labels=['lat','lon',INP])
aux_pred = model.predict(ds_pred,coords=['lat','lon'],**pred_kwargs)

In [ ]:
# Check you have lat,lon now in your output
aux_pred 